![NWM](img/NWM.png)

# Analyze National Water Model Snow Data at Specific Points of Interest
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org)
Last updated:Oct 16, 2025

**Introduction**: This notebook demonstrates how to perform a direct comparison between modeled and observed SWE at individual SNOTEL sites. It uses National Water Model and Snotel data that is collected in the `01_data_collection.ipynb` notebook.

### 1. Prepare the Python Environment

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [17]:
import os
import sys
import glob
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import geopandas as gpd

# Import the Evaluation library from the project root.
sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))
from cssi_evaluation.snow import utils

# import notebook-specific utilities
from utils import nwm_utils

hv.extension('bokeh')

### 2. Build the Input DataFrame

This workflow requires us to organize our observed and modeled snow water equivilent data into a single Pandas DataFrame. We'll do this by reading the data collected in the `01_data_collection.ipynb` notebook.

In [18]:
# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

In [19]:
# gather all the paths to all of the observation and modeled SWE CSV files
obs = glob.glob(os.path.join('obs_outputs', '*.csv'))
mod = glob.glob(os.path.join('mod_outputs', '*.csv'))

In [20]:
# Combine these into a single dataframe using a utilitiy function from the nwm_utils library
combined_df = nwm_utils.combine(obs, mod, StartDate, EndDate)
combined_df

,CCSS_SLI_swe_m,CCSS_KIB_swe_m,CCSS_HRS_swe_m,CCSS_WHW_swe_m,CCSS_DAN_swe_m,CCSS_TES_swe_m,CCSS_TUM_swe_m,CCSS_PDS_swe_m,NWM_DAN_swe_m,NWM_SLI_swe_m,NWM_HRS_swe_m,NWM_TUM_swe_m,NWM_PDS_swe_m,NWM_KIB_swe_m,NWM_TES_swe_m,NWM_WHW_swe_m
2018-10-01,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2018-10-02,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2018-10-03,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2018-10-04,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0114,0.0,0.0,0.0,0.0,0.005,0.0
2018-10-05,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0083,0.0,0.0,0.0,0.0,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-09-26,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2020-09-27,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2020-09-28,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0
2020-09-29,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.000,0.0


### 3. Compare modeled SWE with observed at a site

Plot the GIS data associated with the records in this DataFrame.

In [22]:
# # Path to the watershed shapefile
watershed = "./domain_data/TolumneRiver_18040009.shp"

# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Extract the bounding box coordinates of a watershed
watershed_gdf = gpd.read_file(os.path.join(os.getcwd(), watershed)).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')


m = nwm_utils.plot_sites_within_domain(gdf_in_bbox, watershed_gdf, zoom_start=9)
m

There are 8 sites from CCSS network(s) within the watershed.


In [23]:
gdf_in_bbox

,code,name,network,elevation_m,latitude,longitude,state,HUC,mgrs,mountainRange,beginDate,endDate,csvData,geometry
0,WHW,White Wolf,CCSS,2407.920,37.859445,-119.651607,California,180400090601,11SKB,Sierra Nevada,2007-10-25,2026-01-14,True,POINT (-119.65161 37.85945)
1,TUM,Tuolumne Meadows,CCSS,2621.280,37.876406,-119.348096,California,180400090102,11SKB,Sierra Nevada,2004-10-01,2026-01-14,True,POINT (-119.34810 37.87641)
2,KIB,Lower Kibbie Ridge,CCSS,2042.160,38.033024,-119.879245,California,180400090303,11SKC,Sierra Nevada,2005-10-01,2026-01-14,True,POINT (-119.87924 38.03302)
3,TES,Tioga Pass Entry Station,CCSS,3031.236,37.910870,-119.258507,California,180400090102,11SLB,Sierra Nevada,2004-10-01,2026-01-13,True,POINT (-119.25851 37.91087)
4,DAN,Dana Meadows,CCSS,2987.040,37.896162,-119.257260,California,180400090102,11SLB,Sierra Nevada,2004-10-01,2026-01-14,True,POINT (-119.25726 37.89616)
5,HRS,Horse Meadow,CCSS,2560.320,38.158000,-119.662000,California,180400090402,11SKC,Sierra Nevada,2004-10-01,2026-01-14,True,POINT (-119.66200 38.15800)
6,PDS,Paradise Meadow,CCSS,2331.720,38.046117,-119.671748,California,180400090503,11SKC,Sierra Nevada,2005-10-01,2026-01-14,True,POINT (-119.67175 38.04612)
7,SLI,Slide Canyon,CCSS,2804.160,38.091234,-119.431881,California,180400090501,11SKC,Sierra Nevada,2005-10-01,2026-01-14,True,POINT (-119.43188 38.09123)


Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Once you’ve identified the site of interest, enter its site code in the next code cell.

In [26]:
# choose a gage of interest within the watershed
my_site_code = 'HRS'

In [27]:
gdf_in_bbox[gdf_in_bbox['code']=='HRS']

,code,name,network,elevation_m,latitude,longitude,state,HUC,mgrs,mountainRange,beginDate,endDate,csvData,geometry
5,HRS,Horse Meadow,CCSS,2560.32,38.158,-119.662,California,180400090402,11SKC,Sierra Nevada,2004-10-01,2026-01-14,True,POINT (-119.66200 38.15800)


Plot both datasets.

In [28]:
nwm_utils.comparison_plots(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

:Layout
   .Overlay.I  :Overlay
      .Curve.Observed_SWE :Curve   [index]   (CCSS_HRS_swe_m)
      .Curve.Modeled_SWE  :Curve   [index]   (NWM_HRS_swe_m)
   .Overlay.II :Overlay
      .Scatter.I              :Scatter   [CCSS_HRS_swe_m]   (NWM_HRS_swe_m,index)
      .Curve.A_1_colon_1_Line :Curve   [x]   (y)

Compute the Same-Day SWE Comparison. This is a comparison of the maximum observed SWE for a given year and the corresponding modeled SWE at the time of this occurence. 

**TODO:** Insert a statement explaining this better, it should highligh what this comparison is used for and what it tells us about the modeled data.

In [29]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_cols = sorted([col for col in combined_df.columns if col.startswith('CCSS')])
mod_cols = sorted([col for col in combined_df.columns if col.startswith('NWM')])

In [30]:
# compute the same-day SWE comparison for each of the observed and modeled sites.
df_same_day = utils.same_day_swe_comparison(combined_df, obs_cols, mod_cols)
df_same_day

Skipping (CCSS_TES_swe_m, NWM_TES_swe_m) because all data is NaN


,Observed,Modeled,Water_Year,Station
2019-04-18,0.98044,1.0293,2019,CCSS_DAN_swe_m
2020-04-21,0.41910,0.5160,2020,CCSS_DAN_swe_m
2019-04-20,2.12090,1.3598,2019,CCSS_HRS_swe_m
2020-04-10,0.89662,0.5745,2020,CCSS_HRS_swe_m
2019-03-28,0.80264,0.6708,2019,CCSS_KIB_swe_m
2020-04-09,0.24638,0.1694,2020,CCSS_KIB_swe_m
2019-04-07,1.78562,0.9965,2019,CCSS_PDS_swe_m
2020-04-09,0.68580,0.1688,2020,CCSS_PDS_swe_m
2019-04-19,1.62052,1.4630,2019,CCSS_SLI_swe_m
2020-04-12,0.66802,0.6169,2020,CCSS_SLI_swe_m


Visualize these data graphically.

In [31]:
# Create the scatter plot
scatter_plot_same = df_same_day.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (m)',
    ylabel='Modeled SWE (m)',
    title='SWE on the observed peak SWE date',
    size=15,
    width=400,
    height=300,
    color='#0072B2'
).relabel('Same-Day Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_same_day.drop(columns=['Water_Year']).select_dtypes(include='number').max().max()
one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_same * one_to_one_line).opts(
    legend_position='top'
)

scatter_with_line

:Overlay
   .Scatter.Same_hyphen_minus_Day_Peak_SWE :Scatter   [Observed]   (Modeled)
   .Curve.A_1_colon_1_Line                 :Curve   [x]   (y)

Compute the Different-Day SWE Comparison. This is a comparison of the maximum observed and modeled SWEs for a given year and their corresponding dates of occurence. 

**TODO:** Insert a statement explaining this better, it should highligh what this comparison is used for and what it tells us about the modeled data.

In [32]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_diff_day = utils.different_day_swe_comparison(combined_df, obs_cols, mod_cols)
df_diff_day

Skipping (CCSS_TES_swe_m, NWM_TES_swe_m) because all data is NaN


,Observed,Water_Year,Observed_Date,Modeled,Modeled_Date,Station
0,0.98044,2019,2019-04-18,1.0393,2019-04-10,CCSS_DAN_swe_m
1,0.41910,2020,2020-04-21,0.5206,2020-04-18,CCSS_DAN_swe_m
2,2.12090,2019,2019-04-20,1.5498,2019-04-03,CCSS_HRS_swe_m
3,0.89662,2020,2020-04-10,0.5745,2020-04-10,CCSS_HRS_swe_m
4,0.80264,2019,2019-03-28,0.7210,2019-03-10,CCSS_KIB_swe_m
5,0.24638,2020,2020-04-09,0.2351,2020-01-19,CCSS_KIB_swe_m
6,1.78562,2019,2019-04-07,1.1200,2019-03-10,CCSS_PDS_swe_m
7,0.68580,2020,2020-04-09,0.3302,2020-01-27,CCSS_PDS_swe_m
8,1.62052,2019,2019-04-19,1.4640,2019-04-17,CCSS_SLI_swe_m
9,0.66802,2020,2020-04-12,0.6447,2020-04-21,CCSS_SLI_swe_m


Visualize these data graphically.

In [33]:
# Create the scatter plot
scatter_plot_diff = df_diff_day.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (m)',
    ylabel='Modeled SWE (m)',
    title='Peak SWE Magnitude Comparison',
    size=15,
    width=400,
    height=300,
    color='#E69F00'
).relabel('Different-Day Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_diff_day.drop(columns=['Water_Year']).select_dtypes(include='number').max().max()
one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_same * scatter_plot_diff * one_to_one_line).opts(
    legend_position='top'
)

scatter_with_line

:Overlay
   .Scatter.Same_hyphen_minus_Day_Peak_SWE      :Scatter   [Observed]   (Modeled)
   .Scatter.Different_hyphen_minus_Day_Peak_SWE :Scatter   [Observed]   (Modeled)
   .Curve.A_1_colon_1_Line                      :Curve   [x]   (y)

In [34]:
df_diff_day['Peak_Date_Diff_Days'] = (df_diff_day['Modeled_Date'] - 
                                      df_diff_day['Observed_Date']).dt.days
df_diff_day['SWE_Diff'] = (df_diff_day['Modeled'] - 
                           df_diff_day['Observed'])

df_diff_day.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title='Difference Between Observed and Modeled Peak SWE Dates',
    width=700,
    height=400,
    color='Peak_Date_Diff_Days'
)


:Bars   [Station]   (Peak_Date_Diff_Days)

In [35]:


scatter = df_diff_day.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='SWE_Diff',
    by='Station',
    xlabel='Date Difference (days)',
    ylabel='SWE Difference (m)',
    title='SWE Error vs. Date Shift Between Observed and Modeled Peaks',
    size=80,
    width=600,
    height=400
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='top_left')


:Overlay
   .NdOverlay.I :NdOverlay   [Station]
      :Scatter   [Peak_Date_Diff_Days]   (SWE_Diff)
   .VLine.I     :VLine   [x,y]
   .HLine.I     :HLine   [x,y]

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization helps reveal whether there are **seasonal patterns** in the relationship between observed and modeled SWE. In fact, it allows us to see if the model performs differently during key periods, such as the <span style="color: teal;">**accumulation**</span> phase or the <span style="color: tomato;">**melt**</span> period. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season, which is critical for understanding how well the model represents key processes driving snowpack evolution.

You can change the list of months to highlight (for example, 10 for October or 1 for January) to see how highlighting different parts of the year changes the appearance and interpretation of the scatter plot.

In [36]:
combined_df['month'] = combined_df.index.month

plot = nwm_utils.plot_custom_scatter(combined_df, my_site_code, highlight_months=[10, 11, 12, 1])
plot

:Overlay
   .Scatter.I              :Scatter   [CCSS_HRS_swe_m]   (NWM_HRS_swe_m,color,index,month)
   .Curve.A_1_colon_1_Line :Curve   [x]   (y)

Compute statistics

In [37]:
nwm_utils.compute_stats(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

,Mean,Median,Standard Deviation,Variance,Min,Max
observed,0.495832,0.29972,0.638665,0.407893,0.0,2.1209
modeled,0.322133,0.0669,0.444306,0.197408,0.0,1.5498
,,,,,,
Pearson Correlation,0.959917,,,,,
Spearman Correlation,0.956158,,,,,
Bias (Modeled - Observed),-0.173699,,,,,
Nash-Sutcliffe Efficiency (NSE),0.777549,,,,,
Kling-Gupta Efficiency (KGE),0.534232,,,,,


Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

In [38]:
summary_table = nwm_utils.report_max_dates_and_values(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
summary_table

,Data Source,Peak SWE (m),Date of Maximum
0,CCSS_HRS_swe_m,2.1209,2019-04-20
1,NWM_HRS_swe_m,1.5498,2019-04-03


<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What is the modeled SWE on the date when the observed SWE reaches its peak?
    Use the code snippet below to find the answer.
  </p>
  <pre><code class="language-python">
  
    # Find date of the peak SWE from observed data
    date_obs_max = combined_df['CCSS_HRS_swe_m'].idxmax()

    # Get corresponding value of modeled data on that date
    value_mod_at_max_obs = combined_df.loc[date_obs_max, 'NWM_HRS_swe_m']
  </code></pre>
</div>


Compare the average melt rate over the full melt period. 

The following function computes the melt period length by identifying the first date after the peak SWE when SWE drops to zero and remains at zero for at least (`min_zero_days`) consecutive days. This is used to define the end of the melt period. Finally, the function calculates the average melt rate, which represents the rate at which snow disappeared, expressed in meters per day, over the full melt period.

In [39]:
melt_stats_df = utils.compute_melt_period_statistics(combined_df)
melt_stats_df.head()


,Water_Year,Station,Peak_SWE_Date,Peak_SWE_m,Melt_End_Date,Melt_Period_Days,Melt_Rate_m_per_day
0,2019,CCSS_SLI_swe_m,2019-04-19,1.62052,2019-07-10,82,0.019762
1,2019,CCSS_KIB_swe_m,2019-03-28,0.80264,2019-06-09,73,0.010995
2,2019,CCSS_HRS_swe_m,2019-04-20,2.12090,2019-07-16,87,0.024378
3,2019,CCSS_WHW_swe_m,2019-04-08,1.31572,2019-06-26,79,0.016655
4,2019,CCSS_DAN_swe_m,2019-04-18,0.98044,2019-06-26,69,0.014209


In [40]:
observed_melt_period = nwm_utils.compute_melt_period(combined_df[f'CCSS_{my_site_code}_swe_m'])
observed_melt_period

{'peak_date': Timestamp('2019-04-20 00:00:00', freq='D'),
 'peak_swe_m': 2.1209,
 'melt_end_date': Timestamp('2019-07-16 00:00:00', freq='D'),
 'melt_period_days': 87,
 'melt_rate_m/d': 0.024378160919540228}

In [41]:
modeled_melt_period = nwm_utils.compute_melt_period(combined_df[f'NWM_{my_site_code}_swe_m'])
modeled_melt_period

{'peak_date': Timestamp('2019-04-03 00:00:00', freq='D'),
 'peak_swe_m': 1.5498000230938196,
 'melt_end_date': Timestamp('2019-06-30 00:00:00', freq='D'),
 'melt_period_days': 88,
 'melt_rate_m/d': 0.017611363898793406}